# Correction — Preprocessing sans fuite

Cette correction montre un raisonnement possible. Les chiffres peuvent varier si vous modifiez les données ou la graine.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
data = load_iris(as_frame=True); X = data.data.copy(); y = data.target
X['size_group'] = np.where(X['sepal length (cm)'] > 5.8, 'large', 'small')
X.loc[X.index[::17], 'sepal width (cm)'] = np.nan
print(X.dtypes); print(y.value_counts(normalize=True).round(2))
x_train, x_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=3)
numeric = list(data.feature_names); categorical = ['size_group']
prep = ColumnTransformer([('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric), ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), categorical)])
model = make_pipeline(prep, LogisticRegression(max_iter=1000)).fit(x_train, y_train)
print('test accuracy:', model.score(x_test, y_test))
unknown = x_test.iloc[:1].copy(); unknown['size_group'] = 'unseen'
print('unknown category accepted:', model.predict(unknown))
demo_train = pd.Series([1., 2., 4., 5.]); demo_test = pd.Series([100., 120.])
print('train median:', demo_train.median(), '| illegal global median:', pd.concat([demo_train, demo_test]).median())


## Interprétation

Reliez les nombres affichés à la question. Une autre solution peut convenir si elle respecte les contraintes de split, de métrique et de reproductibilité.